# DBSCAN（1996）

- **核心原理**：密度相連成群（Density-Based Spatial Clustering of Applications with Noise）
- **變體**：OPTICS（一次看所有 ε）、HDBSCAN（階層式、可處理不同密度）
- **推薦資料集**：sklearn 合成資料 `make_moons`（兩個互相勾住的半月形）

第十三章的結尾看到 K-Means 的極限：它的群一定是「圍繞中心的凸塊」（Voronoi 分割），對兩個半月形這種**非凸**的群完全沒辦法；而且每個點都一定要被分到某一群，連離群點也不例外。

DBSCAN 換一個角度定義「群」：

> **一群 = 一塊密度夠高、而且連在一起的區域**；密度太低的地方是**雜訊**（noise），不屬於任何一群。

不需要指定群數 $K$，群可以是任何形狀；代價是要指定「多近算鄰居」（$\varepsilon$）與「多密算密」（`min_samples`）。

這一章的重點：**核心點／邊界點／雜訊的定義、群怎麼用「圖的連通」長出來（BFS）、$\varepsilon$ 怎麼選（k-distance 圖）、以及 DBSCAN 怕什麼（不同密度、高維）。**

README 的流程照第十三章的方式調整（非監督、沒有測試集）。這份是**合成資料，有真實標籤**，所以可以用 ARI 檢查 —— 真實應用通常沒有。

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A、3B、4、5、7 節開頭另有各自的符號表或名詞表。

**資料**：`make_moons` 產生的二維點，兩個半月形各一半，加上一點高斯雜訊（`noise`）。標籤 $y$（0／1）只拿來評估。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $n$ | 點的個數 | 500（以載入結果為準） |
| $\varepsilon$ | 鄰域半徑（`eps`） | — |
| $N_\varepsilon(x)$ | $x$ 的 ε-鄰域：距離 $\le \varepsilon$ 的所有點（**包含 $x$ 自己**） | — |
| `min_samples` | 成為核心點所需的鄰居數（含自己） | — |
| 核心點（core） | $\lvert N_\varepsilon(x)\rvert \ge$ `min_samples` | — |
| 邊界點（border） | 不是核心點，但在某個核心點的 ε-鄰域內 | — |
| 雜訊（noise） | 既不是核心點、也不在任何核心點的鄰域內；標籤 −1 | — |
| 密度可達 | 從核心點出發，沿著「核心點的鄰居」一步步走得到 | — |
| k-distance | 每個點到第 $k$ 近的點的距離（含自己時，第 1 近是自己、距離 0） | — |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第四章 | KNN | 距離矩陣一次算完；尺度決定距離；維度詛咒：高維時所有距離都差不多 → 「近」失去意義。 |
| 第十三章 | K-Means | 群 = 圍繞中心的凸塊（兩個中心之間用垂直平分線分界）；每個點都必須歸到某一群；要先指定 $K$；`clustering_metrics`（silhouette、ARI…）。 |
| 第十三章實驗 3 | 同上 | K-Means 在 moons、不同大小、不同密度、斜橢圓上都會錯。 |
| 第 15 章 | GMM | 每一群是一個（可以是斜的）橢圓，給「軟」的機率。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 鄰域**：$N_\varepsilon(x) = \{x' : \lVert x - x'\rVert \le \varepsilon\}$。例：一維點 $(0, 0.5, 1, 5)$、$\varepsilon = 0.6$ → $N(0) = \{0, 0.5\}$、$N(0.5) = \{0, 0.5, 1\}$、$N(5) = \{5\}$。

**2. 圖與連通**：把點當節點、「互為鄰居」當邊，就是一張圖。**連通分量**（connected component）= 互相走得到的一組節點。DBSCAN 的群 ≈ 只沿著「核心點」走出來的連通分量，再加上掛在它們旁邊的邊界點。

**3. 「轉折點」（knee）**：一條由小排到大的曲線，前段平緩、後段突然變陡；轉折點是「離連接頭尾的直線最遠」的那一點（kneedle 的想法）。點到直線的距離：直線過 $P_1$、$P_2$，點 $Q$ 的距離 $= \frac{|(P_2 - P_1) \times (Q - P_1)|}{\lVert P_2 - P_1\rVert}$（二維的外積 $a \times b = a_x b_y - a_y b_x$）。

#### 程式

**4. 距離矩陣與門檻**（第四章）：`D <= eps` 是 $(n, n)$ 的布林矩陣，第 $i$ 列為 True 的位置就是 $N_\varepsilon(x_i)$；`np.flatnonzero(row)` 取出那些位置；`.sum(axis=1)` = 每個點的鄰居數。

**5. 佇列與廣度優先搜尋（BFS）**：
```python
from collections import deque
q = deque([start])          # 待處理的節點
while q:
    p = q.popleft()         # 先進先出
    for j in neighbors[p]:
        if not visited[j]:
            visited[j] = True
            q.append(j)
```
每個節點最多進佇列一次 → 總成本 = 邊的數量。

**6. 標籤 −1 = 雜訊**：sklearn 的慣例。`labels.max() + 1` = 群數（不含雜訊）；`(labels == -1).mean()` = 雜訊比例。

**7. 排序取第 k 小**：`np.sort(D, axis=1)[:, k - 1]` = 每個點到第 $k$ 近的點的距離（含自己，所以第 1 近是 0）。

---
## 1. 資料與探索性資料分析（EDA）

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from learning_map.data import load_moons

X, y = load_moons()
X, y = np.asarray(X, dtype=float), np.asarray(y).astype(int)
n = len(X)
print("X：", X.shape, "；每一類：", np.bincount(y).tolist())
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=10)
ax.set_title("make_moons (colored by true label)")
ax.set_aspect("equal")
plt.show()

In [ ]:
# 點與點之間的距離：最近鄰距離、第 5 近的距離
# 觀察：（大部分點的最近鄰距離大約多少？兩個半月之間「最窄」的地方大約多寬？
#        如果 ε 比「最窄的間隙」還大，兩個半月會怎樣？如果 ε 比「一般的最近鄰距離」還小呢？）
D_full = np.sqrt(np.maximum((X**2).sum(axis=1)[:, None] + (X**2).sum(axis=1)[None, :] - 2 * X @ X.T, 0))
D_sorted = np.sort(D_full, axis=1)  # 第 0 欄是自己（0）
gap = D_full[np.ix_(y == 0, y == 1)].min()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(D_sorted[:, 1], bins=40, alpha=0.6, label="1st nearest neighbour")
ax.hist(D_sorted[:, 4], bins=40, alpha=0.6, label="4th nearest neighbour")
ax.axvline(gap, color="r", ls="--", label=f"smallest gap between the moons = {gap:.3f}")
ax.set_xlabel("distance")
ax.legend()
plt.show()
print("最近鄰距離：中位數", round(float(np.median(D_sorted[:, 1])), 4), "、最大", round(float(D_sorted[:, 1].max()), 4))
print("第 4 近的距離：中位數", round(float(np.median(D_sorted[:, 4])), 4), "、95 百分位", round(float(np.percentile(D_sorted[:, 4], 95)), 4))

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 兩個半月之間最窄的間隙是多少？大部分點的最近鄰距離是多少？兩者差幾倍？
#     這個「間隙 vs 點與點的間距」的比例，決定了 DBSCAN 好不好做。為什麼？
# 答：
#
# (2) 兩個特徵的單位一樣、尺度也差不多，所以這一章不標準化。如果把 x 軸乘上 10，DBSCAN 的「ε 圓」會變成什麼形狀？（3B 實驗 3）
# 答：

---
## 2. Baseline

- **K-Means（$K = 2$）**（第十三章）。
- **凝聚式階層分群（single linkage）**：一開始每個點自成一群，每次把「最近的兩個點分別屬於的兩群」合併，直到剩 $K$ 群。它也是「沿著近的點連起來」，跟 DBSCAN 很像，但沒有「密度」的概念。

#### 本節名詞

- `clustering_metrics(X, labels, y_true)`（第十三章）：ARI、NMI 越接近 1 越好；雜訊點（−1）在內部指標中會被排除，並回報 `noise_ratio`。

In [ ]:
from sklearn.cluster import AgglomerativeClustering, KMeans
from learning_map.metrics import clustering_metrics

base_labels = {
    "KMeans K=2": KMeans(2, n_init=10, random_state=0).fit_predict(X),
    "Agglomerative single-linkage K=2": AgglomerativeClustering(2, linkage="single").fit_predict(X),
    "Agglomerative ward K=2": AgglomerativeClustering(2, linkage="ward").fit_predict(X),
}
baseline = pd.DataFrame({k: clustering_metrics(X, v, y) for k, v in base_labels.items()}).T
display(baseline)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, lab) in zip(axes, base_labels.items()):
    ax.scatter(X[:, 0], X[:, 1], c=lab, cmap="viridis", s=8)
    ax.set_title(name)
    ax.set_aspect("equal")
plt.show()

In [ ]:
# TODO 2（問答）
#
# (1) K-Means 的 ARI 是多少？它把兩個半月切在哪裡？用第十三章 TODO 7 (3)「兩個中心之間的分界」解釋。
# 答：
#
# (2) single linkage 的結果好不好？如果在兩個半月之間加一條稀稀疏疏的「點的橋」，single linkage 會怎樣？
#     （提示：它只看「最近的一對點」—— 這叫 chaining 問題；DBSCAN 用「密度」來避免）
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 鄰域 → 核心點 → 從核心點做 BFS 長出每一群 | 3、4、5 |
| 3B | 實驗：$\varepsilon$ 與 `min_samples`、雜訊點、不同密度、尺度 | — |

### 3A. DBSCAN

1. **鄰域**（TODO 3）：每個點的 ε-鄰域（含自己）與鄰居數。
2. **核心點**（TODO 4）：鄰居數 $\ge$ `min_samples`。
3. **長群**（TODO 5）：依序走過每個點；遇到「還沒有標籤的核心點」就開一個新群，從它做 BFS：
   - 把它的每個還沒有標籤的鄰居都標上這一群、放進佇列；
   - 從佇列拿出來的點**如果是核心點**，再把它的鄰居加進來；**不是核心點（邊界點）就不往外擴**。
4. 最後還沒有標籤的點 = 雜訊（−1）。

為什麼邊界點不往外擴？邊界點附近的點不夠密，如果也從它往外長，兩個群可能經由一串邊界點「連」在一起（single linkage 的 chaining 問題）。

**邊界點的歸屬**：一個邊界點可能同時在兩群的核心點旁邊 —— 它歸給**先**走到它的那一群。sklearn 也是依點的編號順序處理，所以手寫版（依編號順序）會跟 sklearn 完全一樣。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $D$ | 歐氏距離矩陣 | $(n, n)$ | `D` |
| neighbors | 每個點的 ε-鄰域（列號的陣列，含自己） | 長度 $n$ 的串列 | `self.neighbors_` |
| counts | 每個點的鄰居數 | $(n,)$ | `counts` |
| core | 是不是核心點 | $(n,)$ bool | `self.core_mask_` |
| labels | 群號；−1 = 雜訊（還沒處理的點也先是 −1） | $(n,)$ | `labels` |
| cluster_id | 目前在長的是第幾群 | 純量 | `cluster_id` |

#### 程式結構圖

```
db = DBSCANScratch(eps=0.2, min_samples=5).fit(X)
├── _neighborhoods(X)：D → neighbors_（串列）、counts                            ← TODO 3
├── core_mask_ = counts ≥ min_samples                                            ← TODO 4
└── for i in range(n)：labels[i] == −1 且 core_mask_[i] → _expand(i, cluster_id) ← TODO 5（BFS，就地改 labels）
fit 後：labels_ (n,)、core_mask_ (n,)、core_sample_indices_、neighbors_
```

#### 最小例子：TODO 3、4、5（一維的 DBSCAN）

一維 7 個點 $x = (0, 0.3, 0.6, 2.0, 5.0, 5.2, 5.4)$，$\varepsilon = 0.35$、`min_samples` = 2。

1. **正確答案**：手算。鄰域：0→{0, 0.3}、0.3→{0, 0.3, 0.6}、0.6→{0.3, 0.6}、2.0→{2.0}、5.0→{5.0, 5.2}、5.2→{5.0, 5.2, 5.4}、5.4→{5.2, 5.4}。核心：除了 2.0 以外都是。群：{0, 0.3, 0.6} = 群 0、{5.0, 5.2, 5.4} = 群 1、2.0 = 雜訊。目標 labels = `[0, 0, 0, -1, 1, 1, 1]`。
2. **積木**：`D <= eps`、`np.flatnonzero`、`deque` 的 BFS。
3. **留給你**：寫進 `_neighborhoods`、核心點那一行、`_expand`。

In [ ]:
# 最小例子：TODO 3、4、5（一維的 DBSCAN）
from collections import deque

X_mini = np.array([[0.0], [0.3], [0.6], [2.0], [5.0], [5.2], [5.4]])
eps_mini, ms_mini = 0.35, 2

# ---- 1. 正確答案：照定義一步一步 ----
nb_loop = [[j for j in range(7) if abs(X_mini[i, 0] - X_mini[j, 0]) <= eps_mini] for i in range(7)]
core_loop = [len(v) >= ms_mini for v in nb_loop]
print("鄰域：", nb_loop)
print("核心：", core_loop)

# ---- 2. 積木 ----
D_mini = np.abs(X_mini - X_mini.T)
print("\n(D <= eps) 的第 1 列：", (D_mini <= eps_mini)[1], "→ np.flatnonzero →", np.flatnonzero(D_mini[1] <= eps_mini))
print("(D <= eps).sum(axis=1) =", (D_mini <= eps_mini).sum(axis=1), "← 鄰居數（含自己）")
q_demo = deque([1])
print("deque([1]).popleft() =", q_demo.popleft(), "；佇列現在是空的：", len(q_demo) == 0)

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應為 [0 0 0 -1 1 1 1]）：
# print(DBSCANScratch(eps=eps_mini, min_samples=ms_mini).fit(X_mini).labels_)

In [ ]:
from typing import Self

from numpy.typing import ArrayLike


class DBSCANScratch:
    """DBSCAN（歐氏距離）。labels_：0..k−1 是群、−1 是雜訊。"""

    def __init__(self, eps: float = 0.5, min_samples: int = 5) -> None:
        self.eps = eps
        self.min_samples = min_samples

    def _neighborhoods(self, X: np.ndarray) -> tuple[list[np.ndarray], np.ndarray]:
        """回傳 (neighbors：每個點的 ε-鄰域列號（含自己）, counts (n,))。"""
        D = np.sqrt(np.maximum((X**2).sum(axis=1)[:, None] + (X**2).sum(axis=1)[None, :] - 2 * X @ X.T, 0))
        # TODO 3（程式）：鄰域與鄰居數
        #   - within = D <= self.eps            （(n, n) bool）
        #   - neighbors = [np.flatnonzero(row) for row in within]；counts = within.sum(axis=1)
        raise NotImplementedError("TODO 3：鄰域")
        return neighbors, counts

    def _expand(self, start: int, cluster_id: int, labels: np.ndarray) -> None:
        """從核心點 start 做 BFS，把密度可達的點都標成 cluster_id（就地修改 labels）。"""
        labels[start] = cluster_id
        q = deque([start])
        # TODO 5（程式）：BFS
        #   - while q：p = q.popleft()
        #   - p 不是核心點（self.core_mask_[p] 為 False）→ continue（邊界點不往外擴）
        #   - 否則：for j in self.neighbors_[p]：labels[j] == −1 的就標成 cluster_id、append 進佇列
        raise NotImplementedError("TODO 5：BFS")

    def fit(self, X: ArrayLike) -> Self:
        X = np.asarray(X, dtype=float)
        self.neighbors_, counts = self._neighborhoods(X)
        # TODO 4（程式）：核心點 = 鄰居數（含自己）≥ min_samples
        raise NotImplementedError("TODO 4：核心點")
        labels = np.full(len(X), -1)
        cluster_id = 0
        for i in range(len(X)):
            if labels[i] == -1 and self.core_mask_[i]:
                self._expand(i, cluster_id, labels)
                cluster_id += 1
        self.labels_ = labels
        self.core_sample_indices_ = np.flatnonzero(self.core_mask_)
        return self

    def fit_predict(self, X: ArrayLike) -> np.ndarray:
        return self.fit(X).labels_


from sklearn.cluster import DBSCAN

rows = []
for eps in [0.05, 0.1, 0.15, 0.2, 0.3]:
    for ms in [3, 5, 10]:
        mine = DBSCANScratch(eps, ms).fit(X)
        sk = DBSCAN(eps=eps, min_samples=ms).fit(X)
        rows.append({"eps": eps, "min_samples": ms, "labels identical": np.array_equal(mine.labels_, sk.labels_),
                     "core identical": np.array_equal(mine.core_sample_indices_, sk.core_sample_indices_),
                     "n_clusters": mine.labels_.max() + 1, "noise ratio": np.mean(mine.labels_ == -1)})
cmp = pd.DataFrame(rows)
print("✅ 15 組參數下，labels 與核心點都跟 sklearn 一模一樣：", bool(cmp["labels identical"].all() and cmp["core identical"].all()))
display(cmp.pivot_table(index="eps", columns="min_samples", values=["n_clusters", "noise ratio"]).round(3))

In [ ]:
# 核心點、邊界點、雜訊（eps = 0.1、min_samples = 5：故意選小一點，三種點都會出現）
db = DBSCANScratch(eps=0.1, min_samples=5).fit(X)
border = (db.labels_ != -1) & ~db.core_mask_
noise = db.labels_ == -1
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X[db.core_mask_, 0], X[db.core_mask_, 1], c=db.labels_[db.core_mask_], cmap="tab10", s=14, label="core")
ax.scatter(X[border, 0], X[border, 1], c=db.labels_[border], cmap="tab10", s=40, marker="^", edgecolors="k", label="border")
ax.scatter(X[noise, 0], X[noise, 1], c="k", s=30, marker="x", label="noise")
ax.legend()
ax.set_aspect("equal")
ax.set_title(f"eps = 0.1, min_samples = 5: {db.labels_.max() + 1} clusters, {noise.sum()} noise, {border.sum()} border")
plt.show()

In [ ]:
# TODO 6（問答）DBSCAN 的定義
#
# (1) 最小例子：
#     (a) 2.0 為什麼是雜訊？如果 min_samples 改成 1，它會變成什麼？
#     (b) 如果 ε 改成 0.3（剛好等於 0 與 0.3 的距離），結果會變嗎？「≤」和「<」的差別在這裡重要嗎？
# 答：
#
# (2) 上圖（eps = 0.1）：
#     (a) 有幾群、幾個雜訊、幾個邊界點？群為什麼比 2 多？（看圖：半月被切斷的地方點比較稀還是密？）
#     (b) 邊界點通常落在群的哪裡？
#     (c) 一個邊界點如果同時在兩群的核心點旁邊，它會歸到哪一群？結果跟「點的順序」有關嗎？核心點的分群跟順序有關嗎？
# 答：
#
# (3) 成本：
#     (a) 本章的寫法先算整個 n×n 距離矩陣：時間、記憶體各 O(?)？
#     (b) BFS 本身：每個點最多進佇列幾次？總成本跟什麼成正比？
#     (c) 第四章的 KD-Tree 在低維時可以快速找「半徑 ε 內的點」。sklearn 的 DBSCAN 用的就是它（algorithm="auto"）。
# 答：

### 3B. 實驗：$\varepsilon$ 與 `min_samples`、雜訊點、不同密度、尺度

#### 本節名詞

| 名詞 | 定義 |
| --- | --- |
| 均勻雜訊 | 在資料的範圍內均勻撒的點，模擬「不屬於任何群」的資料 |
| 不同密度 | 一群很密、一群很稀：一個 ε 很難同時適合兩者 |

In [ ]:
# 實驗 1：eps × min_samples 的全面掃描（ARI vs 真實標籤、群數、雜訊比例）
eps_grid = np.round(np.arange(0.04, 0.42, 0.02), 2)
ms_grid = [2, 3, 5, 8, 12, 20]
from sklearn.metrics import adjusted_rand_score

rows = []
for eps in eps_grid:
    for ms in ms_grid:
        lab = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
        rows.append({"eps": eps, "min_samples": ms, "ARI": adjusted_rand_score(y, lab), "n_clusters": lab.max() + 1, "noise": np.mean(lab == -1)})
grid = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, col, fmt in zip(axes, ["ARI", "n_clusters", "noise"], [".2f", ".0f", ".2f"]):
    sns.heatmap(grid.pivot_table(index="min_samples", columns="eps", values=col), annot=True, fmt=fmt, cmap="viridis", ax=ax, annot_kws={"size": 6}, cbar=False)
    ax.set_title(col)
fig.tight_layout()
plt.show()

In [ ]:
# 實驗 2：加入 10% 的均勻雜訊點；實驗 3：不同密度；實驗 4：把 x 軸放大 5 倍
rng = np.random.default_rng(0)
U = rng.uniform(X.min(axis=0) - 0.3, X.max(axis=0) + 0.3, size=(n // 10, 2))
X_noise = np.vstack([X, U])
y_noise = np.r_[y, np.full(len(U), -1)]

from sklearn.datasets import make_blobs

X_dens, y_dens = make_blobs(n_samples=[300, 300], centers=[[0, 0], [4, 0]], cluster_std=[0.2, 1.2], random_state=0)
X_scaled = X * np.array([5.0, 1.0])

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
cases = [
    ("moons + 10% uniform noise", X_noise, DBSCAN(eps=0.15, min_samples=5).fit_predict(X_noise), KMeans(2, n_init=10, random_state=0).fit_predict(X_noise)),
    ("dense blob + sparse blob (eps=0.3)", X_dens, DBSCAN(eps=0.3, min_samples=5).fit_predict(X_dens), DBSCAN(eps=0.8, min_samples=5).fit_predict(X_dens)),
    ("x-axis stretched ×5 (eps=0.15)", X_scaled, DBSCAN(eps=0.15, min_samples=5).fit_predict(X_scaled), DBSCAN(eps=0.15, min_samples=5).fit_predict(X_scaled / X_scaled.std(axis=0) * X.std(axis=0))),
]
titles2 = ["K-Means K=2", "same data, eps=0.8", "after standardizing back"]
for j, (name, Xc, lab1, lab2) in enumerate(cases):
    for i, (lab, t) in enumerate([(lab1, f"DBSCAN: {name}"), (lab2, titles2[j])]):
        ax = axes[i, j]
        ax.scatter(Xc[lab >= 0, 0], Xc[lab >= 0, 1], c=lab[lab >= 0], cmap="tab10", s=6)
        ax.scatter(Xc[lab < 0, 0], Xc[lab < 0, 1], c="k", marker="x", s=10)
        ax.set_title(f"{t}\nclusters={lab.max() + 1}, noise={np.mean(lab == -1):.2f}", fontsize=9)
fig.tight_layout()
plt.show()

lab_n = DBSCAN(eps=0.15, min_samples=5).fit_predict(X_noise)
print(f"實驗 2：真的雜訊點被標成 −1 的比例 {np.mean(lab_n[len(X):] == -1):.2f}；原本的月亮點被誤標成 −1 的比例 {np.mean(lab_n[: len(X)] == -1):.3f}")

In [ ]:
# TODO 7（問答）參數與限制
#
# (1) 實驗 1：
#     (a) ARI ≈ 1 的區域在哪裡（eps 大約多少到多少）？太小、太大各發生什麼事（看 n_clusters、noise）？
#     (b) min_samples 變大時，好的 eps 範圍往哪邊移？為什麼？（提示：要更多鄰居才算核心 → 圓要大一點才裝得到）
#     (c) 跟 K-Means 的 K 比，DBSCAN 的兩個參數比較好調還是比較難調？
# 答：
#
# (2) 實驗 2：
#     (a) DBSCAN 把多少比例的均勻雜訊點標成 −1？K-Means 對這些點做了什麼？
#     (b) 有沒有雜訊點被併進某個月亮？為什麼（看它的位置）？
# 答：
#
# (3) 實驗 3（不同密度）：
#     (a) eps = 0.3 時，稀疏那一群發生什麼事？eps = 0.8 呢？有沒有一個 eps 能同時照顧兩群？
#     (b) 這就是 DBSCAN 最大的弱點。第 4 節的 HDBSCAN 怎麼處理？
# 答：
#
# (4) 實驗 4（尺度）：x 軸放大 5 倍之後，同一個 eps 的結果變成怎樣？為什麼？（ε 圓在原本的座標裡變成什麼形狀？）
# 答：

---
## 4. 套件實作：`DBSCAN`、`OPTICS`、`HDBSCAN`

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| `DBSCAN(eps, min_samples, metric, algorithm)` | 本章的演算法；`algorithm="auto"` 低維時用 KD-Tree／Ball Tree 找鄰域 |
| `OPTICS` | 不固定 ε：把點排成一個「可達距離」的順序，任何 ε 的 DBSCAN 結果都可以從這個排序讀出來 |
| `HDBSCAN` | 在所有 ε 上建一棵「群的階層樹」，挑出最穩定的群；不同密度的群可以同時被找出來。主要參數只剩 `min_cluster_size` |
| `probabilities_`（HDBSCAN） | 每個點屬於它那一群的「強度」（0–1） |

In [ ]:
from sklearn.cluster import HDBSCAN, OPTICS

methods = {
    "DBSCAN eps=0.15": lambda: DBSCAN(eps=0.15, min_samples=5),
    "OPTICS min_samples=10": lambda: OPTICS(min_samples=10),
    "HDBSCAN min_cluster_size=20": lambda: HDBSCAN(min_cluster_size=20, copy=True),
}
datasets = {"moons": (X, y), "moons + uniform noise": (X_noise, y_noise), "dense + sparse blobs": (X_dens, y_dens)}
rows = []
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for i, (dname, (Xd, yd)) in enumerate(datasets.items()):
    for j, (mname, make) in enumerate(methods.items()):
        lab = make().fit_predict(Xd)
        keep = yd >= 0  # 雜訊點沒有真實的群，ARI 只在真正的群上算
        rows.append({"data": dname, "method": mname, "ARI (real points)": adjusted_rand_score(yd[keep], lab[keep]), "n_clusters": lab.max() + 1, "noise": np.mean(lab == -1)})
        ax = axes[i, j]
        ax.scatter(Xd[lab >= 0, 0], Xd[lab >= 0, 1], c=lab[lab >= 0], cmap="tab10", s=5)
        ax.scatter(Xd[lab < 0, 0], Xd[lab < 0, 1], c="k", marker="x", s=8)
        ax.set_title(f"{mname} | {dname}", fontsize=9)
fig.tight_layout()
plt.show()
display(pd.DataFrame(rows).set_index(["data", "method"]).round(3))

In [ ]:
# TODO 8（問答）
#
# (1) 三種方法在三份資料上，哪一個整體最好？HDBSCAN 在「不同密度」的資料上有沒有比 DBSCAN 好？
# 答：
#
# (2) HDBSCAN 的主要參數 min_cluster_size（最小的群要多大）比 DBSCAN 的 eps 好設定在哪裡？
#     （提示：eps 的單位是「距離」，跟資料的尺度綁在一起；min_cluster_size 的單位是什麼？）
# 答：

---
## 5. 選參數：k-distance 圖

真實資料沒有標籤可以算 ARI。常用的經驗法則（Ester et al. 1996）：

1. 先定 `min_samples`（二維常用 4–5；維度越高越大，例如 $2d$）；
2. 算每個點到「第 `min_samples` 近」的點的距離（含自己），由小排到大畫出來（**k-distance 圖**）；
3. 曲線前段平緩（群內的點，距離都差不多小）、後段突然變陡（雜訊點，離誰都遠）。**轉折點**的距離就是 ε 的候選。

找轉折點（TODO 9）：把曲線的頭、尾連成一條直線，找**離這條直線最遠**的點（前置複習 3）。

#### 符號表（k-distance）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| kd | 排序後的 k-distance | $(n,)$ | `kd` |
| $P_1$、$P_2$ | 曲線的頭 $(0, kd_0)$、尾 $(n-1, kd_{n-1})$ | $(2,)$ | `p1`、`p2` |
| $Q_i$ | 曲線上第 $i$ 點 $(i, kd_i)$ | $(n, 2)$ | `Q` |
| dist | 每個 $Q_i$ 到直線 $P_1P_2$ 的距離 | $(n,)$ | `dist` |

座標要先縮放到 $[0, 1]$（x 軸是點的編號 0..n−1、y 軸是距離，單位完全不同，不縮放的話「最遠」沒有意義）。

#### 最小例子：TODO 9（找轉折點）

曲線 `kd = [1, 1.1, 1.2, 1.3, 1.5, 3, 6]`（已排序）。

1. **正確答案**：縮放後逐點算到首尾連線的距離，最大的是第 4 點（kd = 1.5）。
2. **積木**：縮放到 [0, 1]、二維外積 $a_x b_y - a_y b_x$。
3. **留給你**：向量化算出所有點的距離、`argmax`。

In [ ]:
# 最小例子：TODO 9（找轉折點）
kd_mini = np.array([1, 1.1, 1.2, 1.3, 1.5, 3, 6], dtype=float)

# ---- 1. 正確答案：逐點 ----
xs = np.arange(len(kd_mini)) / (len(kd_mini) - 1)
ys = (kd_mini - kd_mini.min()) / (kd_mini.max() - kd_mini.min())
p1_, p2_ = np.array([xs[0], ys[0]]), np.array([xs[-1], ys[-1]])
dist_loop = []
for i in range(len(kd_mini)):
    a, b = p2_ - p1_, np.array([xs[i], ys[i]]) - p1_
    dist_loop.append(abs(a[0] * b[1] - a[1] * b[0]) / np.linalg.norm(a))
print("距離 =", np.round(dist_loop, 3), "→ 轉折點是第", int(np.argmax(dist_loop)), "點，kd =", kd_mini[int(np.argmax(dist_loop))])

# ---- 2. 積木 ----
Q_mini = np.column_stack([xs, ys])
print("\nQ − P1 的 shape：", (Q_mini - p1_).shape, "；外積 a_x·b_y − a_y·b_x 對每一列：", ((p2_ - p1_)[0] * (Q_mini - p1_)[:, 1] - (p2_ - p1_)[1] * (Q_mini - p1_)[:, 0]).round(3))

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應為 4）：
# print(knee_index(kd_mini))

In [ ]:
def knee_index(kd: np.ndarray) -> int:
    """排序後的曲線 kd (n,) → 轉折點的位置：縮放到 [0, 1] 後，離首尾連線最遠的點。"""
    kd = np.asarray(kd, dtype=float)
    m = len(kd)
    Q = np.column_stack([np.arange(m) / (m - 1), (kd - kd.min()) / (kd.max() - kd.min())])  # (m, 2)
    p1, p2 = Q[0], Q[-1]
    # TODO 9（程式）
    #   - a = p2 − p1 (2,)；B = Q − p1 (m, 2)
    #   - cross = a[0]·B[:, 1] − a[1]·B[:, 0]；dist = |cross| / ‖a‖
    #   - 回傳 argmax(dist)
    raise NotImplementedError("TODO 9：轉折點")


from sklearn.neighbors import NearestNeighbors

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
chosen = {}
for ax, (dname, Xd) in zip(axes, [("moons", X), ("moons + uniform noise", X_noise)]):
    for ms in [3, 5, 10]:
        dist_k, _ = NearestNeighbors(n_neighbors=ms).fit(Xd).kneighbors(Xd)  # 含自己：第 1 欄是 0
        kd = np.sort(dist_k[:, -1])
        k_i = knee_index(kd)
        chosen[(dname, ms)] = kd[k_i]
        ax.plot(kd, label=f"min_samples = {ms} → eps ≈ {kd[k_i]:.3f}")
        ax.plot(k_i, kd[k_i], "ko")
    ax.set_xlabel("points sorted by k-distance")
    ax.set_ylabel("distance to k-th neighbour")
    ax.set_title(dname)
    ax.legend()
plt.show()

rows = []
for (dname, ms), eps in chosen.items():
    Xd, yd = datasets[dname]
    lab = DBSCAN(eps=eps, min_samples=ms).fit_predict(Xd)
    keep = yd >= 0
    rows.append({"data": dname, "min_samples": ms, "eps from knee": eps, "ARI (real points)": adjusted_rand_score(yd[keep], lab[keep]), "n_clusters": lab.max() + 1, "noise": np.mean(lab == -1)})
display(pd.DataFrame(rows).round(3))

In [ ]:
# TODO 10（問答）k-distance 圖
#
# (1) 兩份資料的 k-distance 圖，「突然變陡」的地方在哪裡？加了雜訊的那份為什麼尾巴特別高？
# 答：
#
# (2) 用轉折點選出來的 eps，得到的 ARI 跟實驗 1 的最佳區域比如何？轉折點法是在找什麼樣的 eps？
#     （提示：它讓「大部分」點都能當核心點，只留下尾巴那些離誰都遠的點 —— 跟「兩個月亮之間的間隙」有關嗎？）
# 答：
#
# (3) 沒有雜訊的 moons、min_samples = 3 或 5 時，轉折點選出來的 eps 太小（群被切碎）；加了均勻雜訊的資料反而選得很好。為什麼？
#     （提示：k-distance 圖的「陡峭尾巴」是誰造成的？沒有雜訊時，尾巴是哪些點？—— 月亮上比較稀疏的那一段）
#     所以轉折點法隱含了什麼假設？
# 答：
#
# (4) 如果兩群的密度差很多（實驗 3），k-distance 圖會有幾段平緩的地方？一個轉折點夠嗎？
# 答：

---
## 6. 評估

`clustering_metrics(X, labels, y_true)`（雜訊點在內部指標中排除、回報 `noise_ratio`）；`log_result`／`compare` 寫進 `results/moons.csv`，依 ARI 排序。

In [ ]:
from learning_map.results import compare, log_result

BENCH = "moons"
eps_knee = chosen[("moons", 5)]
runs = {
    "KMeans K=2": (base_labels["KMeans K=2"], {}),
    "Agglomerative single-linkage K=2": (base_labels["Agglomerative single-linkage K=2"], {}),
    "DBSCAN eps from knee, min_samples=5 (scratch)": (DBSCANScratch(eps_knee, 5).fit_predict(X), {"eps": round(float(eps_knee), 4), "min_samples": 5}),
    "DBSCAN eps=0.15, min_samples=5 (sklearn)": (DBSCAN(eps=0.15, min_samples=5).fit_predict(X), {"eps": 0.15, "min_samples": 5}),
    "HDBSCAN min_cluster_size=20 (sklearn)": (HDBSCAN(min_cluster_size=20, copy=True).fit_predict(X), {"min_cluster_size": 20}),
}
scores = {}
for name, (lab, params) in runs.items():
    m = clustering_metrics(X, lab, y)
    scores[name] = m
    log_result(BENCH, name, m, **params)
display(pd.DataFrame(scores).T)
compare(BENCH, "ari")

---
## 7. 診斷

看 knee 選出來的 DBSCAN 錯在哪：被標成雜訊的點、被分到錯的月亮的點，以及它們離最近的核心點有多遠。

In [ ]:
final = DBSCANScratch(eps_knee, 5).fit(X)
lab = final.labels_
# 把群號對應到真實標籤（多數決），才能數「分錯」
mapping = {k: np.bincount(y[lab == k]).argmax() for k in range(lab.max() + 1)}
pred = np.array([mapping.get(k, -1) for k in lab])
wrong = (pred != y) & (lab != -1)
noise = lab == -1
d_core = np.sqrt(((X[:, None, :] - X[final.core_mask_][None, :, :]) ** 2).sum(axis=-1)).min(axis=1)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X[:, 0], X[:, 1], c=lab, cmap="tab10", s=8, alpha=0.5)
ax.scatter(X[noise, 0], X[noise, 1], c="k", marker="x", s=40, label=f"noise ({noise.sum()})")
ax.scatter(X[wrong, 0], X[wrong, 1], facecolors="none", edgecolors="r", s=80, label=f"wrong moon ({wrong.sum()})")
ax.legend()
ax.set_aspect("equal")
ax.set_title(f"DBSCAN (eps = {eps_knee:.3f}, min_samples = 5)")
plt.show()
print("雜訊點到最近核心點的距離：", np.round(np.sort(d_core[noise]), 3), "；eps =", round(float(eps_knee), 3))

In [ ]:
# TODO 11（問答，本章總結）
#
# (1) knee 選出來的 DBSCAN：有幾個雜訊點、幾個分錯月亮的點？雜訊點落在哪裡（月亮的尖端？中間？）
#     它們到最近核心點的距離都比 eps 大嗎？（應該要 —— 為什麼？）
# 答：
#
# (2) DBSCAN vs K-Means 比較表（填空）：
#
#     |                        | K-Means（第十三章） | DBSCAN（本章） |
#     | 要指定群數嗎           |                     |                |
#     | 群的形狀               |                     |                |
#     | 雜訊／離群點           |                     |                |
#     | 不同密度的群           |                     |                |
#     | 對尺度的敏感度         |                     |                |
#     | 結果是否唯一（隨機性） |                     |                |
#     | 新的點怎麼預測         |                     |                |
#
#     （最後一列的提示：sklearn 的 DBSCAN 沒有 predict；要分配新點，可以怎麼做？）
# 答：
#
# (3) 回到第十三章的 Wholesale 資料：如果用 DBSCAN 分群，你預期會得到什麼？（提示：那份資料的 silhouette 很低 —— 群跟群之間有沒有「低密度的間隙」？）
#     你覺得 DBSCAN 適合那份資料嗎？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

make_moons（真實標籤只用於評估）。

| 方法 | 做法 | 關鍵性質 | ARI | 雜訊比例 |
| --- | --- | --- | --- | --- |
| K-Means $K = 2$ | | | | |
| Agglomerative single linkage | | | | |
| DBSCAN（手寫，knee 選 eps） | 鄰域 → 核心點 → BFS | | | |
| DBSCAN（sklearn，eps = 0.15） | | | | |
| HDBSCAN | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**演算法**
- ☐ OPTICS 的 reachability plot 怎麼讀；從 reachability plot 抽出任意 ε 的 DBSCAN 結果
- ☐ HDBSCAN：mutual reachability distance、minimum spanning tree、cluster stability（excess of mass）
- ☐ 用空間索引（KD-Tree、Ball Tree）加速鄰域查詢；高維時的困難（第四章維度詛咒）

**應用與延伸**
- ☐ 地理資料（經緯度用 haversine 距離）的 DBSCAN
- ☐ 用 DBSCAN 做異常偵測：雜訊點 = 異常（對照第八章 Isolation Forest）
- ☐ Mean Shift、Spectral Clustering 也能處理非凸的群
- ☐ 把新點分配到既有的群：最近的核心點、或 HDBSCAN 的 `approximate_predict`